In [99]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import wandb
import os

from pathlib import Path
from torch.optim import Adam
from torch.nn import BCEWithLogitsLoss
from torch.utils.data import DataLoader,TensorDataset
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

## Descargar los datos

In [100]:
!pip install kaggle
!mkdir -p data
!kaggle datasets download -d blastchar/telco-customer-churn -p data --unzip

Dataset URL: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
License(s): copyright-authors
100%|█████████████████████████████████████████| 172k/172k [00:00<00:00, 661kB/s]



## Visualizar los datos

In [101]:
csv_path = '/home/jaume/ConquerX/Mis_apuntes/Pytorch/ejercicio_redes/Teleco_Customer_Churn/data/WA_Fn-UseC_-Telco-Customer-Churn.csv'

In [102]:
df = pd.read_csv(csv_path)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [103]:
df.shape

(7043, 21)

In [104]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [105]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [106]:
df["Churn"].value_counts()

Churn
No     5174
Yes    1869
Name: count, dtype: int64

In [107]:
# Hay que mirar columnas de tipo Object que contengan numeros siemrpe!
# Pueden tener espacios de texto vacios que isnull no contea.

df[df["TotalCharges"] == " "] # Esta es la unica columna con estas caracteristicas

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,,No
753,3115-CZMZD,Male,0,No,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,,No
936,5709-LVOEQ,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,...,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,,No
1082,4367-NUYAO,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,,No
1340,1371-DWPAZ,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,,No
3331,7644-OMVMY,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,,No
3826,3213-VVOLG,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,,No
4380,2520-SGTTA,Female,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,,No
5218,2923-ARZLG,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,,No
6670,4075-WKNIU,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,...,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,,No


## Interpretacion de los datos

### Contexto
"Prediga el comportamiento para fidelizar a sus clientes. Puede analizar todos los datos relevantes de los clientes y desarrollar programas de retención de clientes específicos." [Conjuntos de datos de muestra de IBM]

### Contenido
Cada fila representa a un cliente, y cada columna contiene los atributos del cliente descritos en la columna Metadatos.

### Interpretacion:
A primera vista no existen datos nulos pero si que los hay, hay columnas de tipo "Objeto" con filas vacias, por eso no los detecta como nulos.
La mayoria de las columnas son útiles para el entranmiento
La variable objetivo esta desvalanceada



## Limpieza de datos

In [108]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce") #Convierte la columna a numerica y las filas vacias en Nan

In [109]:
df = df.drop(columns=['customerID']) # Borro columnas
df = df.dropna() # Borro nulos
df = df.drop_duplicates() # Borro duplicados

In [110]:
print(df["TotalCharges"].isnull().sum()) #Comprobacion
print(df.shape)

0
(7010, 20)


In [111]:
df["Churn"] = df["Churn"].map({"No":0,"Yes":1}) # Mapear la variable objetivo

In [112]:
# Separar la variable objetivo del resto

y_dataset = df["Churn"]
x_dataset = df.drop(columns="Churn")

In [113]:
# Separar dataset TRAIN/VAL/TEST

x_train,x_test,y_train,y_test = train_test_split(
    x_dataset,y_dataset, test_size=0.2 ,train_size=0.8,random_state=1,stratify=y_dataset)

x_train,x_val,y_train,y_val = train_test_split(
    x_train,y_train, test_size=0.2 ,train_size=0.8,random_state=1,stratify=y_train)


In [114]:
# Comprobacion
print(x_train.shape)
print(x_val.shape)
print(x_test.shape)
print(y_train.shape)
print(y_val.shape)
print(y_test.shape)

(4486, 19)
(1122, 19)
(1402, 19)
(4486,)
(1122,)
(1402,)


## Transformar los datos

In [115]:
# Separar columnas categoricas y numericas para en OneHotEncoder

categorical_cols = x_train.select_dtypes(include="object").columns
numerical_cols = x_train.select_dtypes(exclude="object").columns

In [116]:
# OneHotEncoding solo a las columnas ncategoricas(para que sean numericas)
encoder = OneHotEncoder(
    handle_unknown="ignore", # Si en validation o test aparece una categoría que no existía en train,no da error.
    sparse_output=False # Devuelve un array normal de NumPy.
)

x_train_encoded = encoder.fit_transform(x_train[categorical_cols])
x_val_encoded = encoder.transform(x_val[categorical_cols])
x_test_encoded = encoder.transform(x_test[categorical_cols])


In [117]:
# Comprobacion
print(x_train_encoded.shape)
print(x_val_encoded.shape)
print(x_test_encoded.shape)

(4486, 41)
(1122, 41)
(1402, 41)


In [118]:
# Escalar las columnas numericas
scaler = StandardScaler()

x_train_num_scaled = scaler.fit_transform(x_train[numerical_cols])
x_val_num_scaled = scaler.transform(x_val[numerical_cols])
x_test_num_scaled = scaler.transform(x_test[numerical_cols])


In [119]:
# Comprobacion
print(x_train_num_scaled.shape)
print(x_val_num_scaled.shape)
print(x_test_num_scaled.shape)

(4486, 4)
(1122, 4)
(1402, 4)


## Convertir los datos en Tensores

In [120]:
# Concateno los arrays procesados para TRAIN/VAL/TEST

x_train_processed = np.concatenate(
    [x_train_num_scaled, x_train_encoded],
    axis=1
)

x_val_processed = np.concatenate(
    [x_val_num_scaled, x_val_encoded],
    axis=1
)

x_test_processed = np.concatenate(
    [x_test_num_scaled, x_test_encoded],
    axis=1
)

In [121]:
# Comprobacion
print(x_train_processed.shape)
print(x_val_processed.shape)
print(x_test_processed.shape)

(4486, 45)
(1122, 45)
(1402, 45)


In [122]:
# Combierto de array a tensor:
t_x_train = torch.from_numpy(x_train_processed).float()
t_x_val = torch.from_numpy(x_val_processed).float()
t_x_test = torch.from_numpy(x_test_processed).float()

t_y_train = torch.from_numpy(y_train.values).float().unsqueeze(1)
t_y_val = torch.from_numpy(y_val.values).float().unsqueeze(1)
t_y_test = torch.from_numpy(y_test.values).float().unsqueeze(1)


print(f"x train: {t_x_train.shape}\nx val: {t_x_val.shape}\nx test: {t_x_test.shape}\n")
print(f"y train: {t_y_train.shape}\ny val: {t_y_val.shape}\ny test: {t_y_test.shape}")

x train: torch.Size([4486, 45])
x val: torch.Size([1122, 45])
x test: torch.Size([1402, 45])

y train: torch.Size([4486, 1])
y val: torch.Size([1122, 1])
y test: torch.Size([1402, 1])


## Configuracion de parametros

In [123]:
config = {
    "input_size":t_x_train.shape[1],
    "hidden_size_1":64,
    "hidden_size_2":32,
    "batch_size":32,
    "lr":0.001,
    "epochs":3,
    "dropout":0.2,
    "weight_decay":0.0001,
    "device":"cpu",
}

In [124]:
sweep_config = {
    "name": "Telco-Customer-Churn",
    "method": "random",

    "metric": {
        "name": "val_f1",
        "goal": "maximize"
    },

    "parameters": {
        "lr": {
            "values": [0.01, 0.001, 0.0001]
        },

        "batch_size": {
            "values": [16, 32, 64]
        },

        "hidden_size_1": {
            "values": [32, 64, 128]
        },

        "hidden_size_2": {
            "values": [16, 32, 64]
        },

        "dropout": {
            "values": [0.0, 0.2, 0.4]
        },

        "weight_decay": {
            "values": [0.0, 0.0001, 0.001]
        },

        "epochs": {
            "values": [10,15,20,25]
        }
    }
}

## Agrupar los Tensores

In [125]:
train_dataset = TensorDataset(t_x_train,t_y_train)
val_dataset = TensorDataset(t_x_val,t_y_val)
test_dataset = TensorDataset(t_x_test,t_y_test)

## Crear los DataLoaders

In [126]:
train_loader = DataLoader(train_dataset,batch_size=config["batch_size"],shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=config["batch_size"],shuffle=False)
test_loader = DataLoader(test_dataset,batch_size=config["batch_size"],shuffle=False)

In [127]:
print(torch.isnan(t_x_train).sum())
print(torch.isnan(t_y_train).sum())

tensor(0)
tensor(0)


## Red Neuronal

In [128]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

In [129]:

class ChurnModel(nn.Module):

    def __init__(self, input_size, hidden_size_1, hidden_size_2, dropout):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size_1),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_size_1, hidden_size_2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_size_2, 1)
        )

    def forward(self, x):
        return self.network(x)

## Funciones

In [130]:
def evaluate(model, data_loader, criterion, device):

    model.eval()

    total_loss = 0
    total_correct = 0
    total_samples = 0
    total_true_positives = 0
    total_false_positives = 0
    total_false_negatives = 0

    all_targets = []
    all_probabilities = []  

    with torch.no_grad():

        for x_batch, y_batch in data_loader:

            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(x_batch)

            loss = criterion(logits, y_batch)

            probabilities = torch.sigmoid(logits)

            all_targets.append(y_batch.cpu())
            
            all_probabilities.append(probabilities.cpu())  

            predictions = (probabilities >= 0.5).float()

            total_loss += loss.item() * x_batch.size(0)

            total_correct += (predictions == y_batch).sum().item()

            total_samples += y_batch.size(0)

            total_true_positives += ((predictions==1)&(y_batch==1)).sum().item()

            total_false_positives += ((predictions==1)&(y_batch==0)).sum().item()

            total_false_negatives += ((predictions==0)&(y_batch==1)).sum().item()
           
    # Calcula la precision
    if total_true_positives + total_false_positives == 0:
        precision = 0
    else:
        precision = total_true_positives / (total_true_positives + total_false_positives)

    # Calcula el racall
    if total_true_positives + total_false_negatives == 0:
        recall = 0
    else:
        recall = total_true_positives / (total_true_positives + total_false_negatives)

    # Calcula F1 Score
    if precision + recall == 0:
        f1 = 0
    else:
        f1 = 2 * (precision * recall) / (precision + recall)


    # Calcula AUC
    all_targets = torch.cat(all_targets).view(-1).numpy()
    all_probabilities = torch.cat(all_probabilities).view(-1).numpy()
    auc = roc_auc_score(all_targets, all_probabilities)
    
    average_loss = total_loss / total_samples

    accuracy = total_correct / total_samples

    return average_loss, accuracy, precision, recall, f1, auc

In [131]:
def train_model(model,train_loader,val_loader,criterion,optimizer,device,epochs,model_path=None,use_WB=False):

        history = {
            "train_loss": [],
            "train_accuracy": [],
            "val_loss": [],
            "val_accuracy": [],
            "val_precision": [],
            "val_recall": [],
            "val_f1": [],
            "val_auc": []
        }

        for epoch in range(epochs):

            model.train()

            total_train_loss = 0
            total_correct = 0
            total_samples = 0

            for x_batch, y_batch in train_loader:

                x_batch = x_batch.to(device) # Carga el batch al dispositivo seleccionado
                y_batch = y_batch.to(device)

                optimizer.zero_grad() # Poner los gradientes a 0

                logits = model(x_batch) # Output del modelo en logits por falta de funcion de activacion en la ultima capa

                loss = criterion(logits, y_batch) # Calcula la perdida 

                loss.backward() 

                optimizer.step() # Actualiza los pesos usando los gradientes calculados

                probabilities = torch.sigmoid(logits) # Calcula las predicciones en probabilidades

                predictions = (probabilities >= 0.5).float() # Convierte las probabilidades en 0 o 1

                total_train_loss += loss.item() * x_batch.size(0) 

                total_correct += (predictions == y_batch).sum().item()

                total_samples += y_batch.size(0)

            train_loss = total_train_loss / total_samples

            train_accuracy = total_correct / total_samples

            val_loss, val_accuracy, val_precision, val_recall, val_f1, val_auc = evaluate(
                model,
                val_loader,
                criterion,
                device
            )

            history["train_loss"].append(train_loss)
            history["train_accuracy"].append(train_accuracy)
            history["val_loss"].append(val_loss)
            history["val_accuracy"].append(val_accuracy)
            history["val_precision"].append(val_precision)
            history["val_recall"].append(val_recall)
            history["val_f1"].append(val_f1)
            history["val_auc"].append(val_auc)

            print(
                f"Epoch {epoch + 1}/{epochs} | "
                f"Train Loss: {train_loss:.4f} | "
                f"Train Accuracy: {train_accuracy:.4f} | "
                f"Val Loss: {val_loss:.4f} | "
                f"Val Accuracy: {val_accuracy:.4f} | "
                f"Val Precision: {val_precision:.4f} | "
                f"Val Recall: {val_recall:.4f} | "
                f"Val F1: {val_f1:.4f} | "
                f"Val AUC: {val_auc:.4f} | "
            )

            if use_WB == True:
                wandb.log({
                    "train_accuracy":train_accuracy,
                    "train_loss":train_loss,
                    "val_loss":val_loss,
                    "val_accuracy":val_accuracy,
                    "val_precision":val_precision,
                    "val_recall":val_recall,
                    "val_f1":val_f1,
                    "val_auc":val_auc,
                })

        return history


In [132]:
def train_sweep(config_2):

    with wandb.init(
        project="Teleco-Customer-Churn",
    ) as run:
        
        config = run.config
        run.name = f"lr_{config.lr}_bsize_{config.batch_size}_dout_{config.dropout}"

        train_loader = DataLoader(train_dataset,batch_size=config.batch_size,shuffle=True)
        val_loader = DataLoader(val_dataset,batch_size=config.batch_size,shuffle=False)
        test_loader = DataLoader(test_dataset,batch_size=config.batch_size,shuffle=False)

        model = ChurnModel(
            config_2["input_size"],
            config.hidden_size_1,
            config.hidden_size_2,
            config.dropout
        ).to(config_2["device"])
        
        criterion = BCEWithLogitsLoss()
        optimizer = Adam(
            model.parameters(),
            lr=config.lr,
            weight_decay=config.weight_decay
        )

        history = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion = criterion,
            optimizer=optimizer,
            epochs= config.epochs,
            device=config_2["device"],
            use_WB=True
        )

        # Guardar los modelos de cada run
        os.makedirs("models", exist_ok=True) #Crea el directorio para los modelos
        model_path = f"models/model_{run.id}.pt" #Crea la ruta del modelo

        config_to_save = dict(config)
        config_to_save["input_size"] = config_2["input_size"]
        
        torch.save({
            "model_state_dict": model.state_dict(),
            "valores_config": dict(config_to_save)
        }, model_path)

        run.summary["model_path"] = model_path #Añade la ruta al resumen de W&B

        return history



In [133]:
# model = ChurnModel(config["input_size"],config["hidden_size_1"],config["hidden_size_2"],config["dropout"])
# criterion = BCEWithLogitsLoss()
# optimizer = Adam(
#     model.parameters(),
#     lr=config["lr"],
#     weight_decay=config["weight_decay"]
# )

In [134]:
# train_model(
#     model=model,
#     train_loader=train_loader,
#     val_loader=val_loader,
#     criterion = criterion,
#     optimizer=optimizer,
#     epochs= config["epochs"],
#     device=config["device"]
# )

In [135]:
wandb.login()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

In [136]:
sweep_id = wandb.sweep(sweep=sweep_config,project="Teleco-Customer-Churn")

Create sweep with ID: 836h91kq
Sweep URL: https://wandb.ai/jpalauserrano-conquer-blocks/Teleco-Customer-Churn/sweeps/836h91kq


In [137]:
wandb.agent(sweep_id=sweep_id, function=lambda:train_sweep(config), count=1)

wandb: Agent Starting Run: 99fldt8x with config:
wandb: 	batch_size: 32
wandb: 	dropout: 0.4
wandb: 	epochs: 10
wandb: 	hidden_size_1: 64
wandb: 	hidden_size_2: 32
wandb: 	lr: 0.0001
wandb: 	weight_decay: 0.0001
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/jaume/.netrc.


Epoch 1/10 | Train Loss: 0.6561 | Train Accuracy: 0.6895 | Val Loss: 0.6162 | Val Accuracy: 0.7353 | Val Precision: 0.0000 | Val Recall: 0.0000 | Val F1: 0.0000 | Val AUC: 0.7452 | 
Epoch 2/10 | Train Loss: 0.5909 | Train Accuracy: 0.7356 | Val Loss: 0.5506 | Val Accuracy: 0.7353 | Val Precision: 0.0000 | Val Recall: 0.0000 | Val F1: 0.0000 | Val AUC: 0.8228 | 
Epoch 3/10 | Train Loss: 0.5347 | Train Accuracy: 0.7367 | Val Loss: 0.4988 | Val Accuracy: 0.7353 | Val Precision: 0.0000 | Val Recall: 0.0000 | Val F1: 0.0000 | Val AUC: 0.8346 | 
Epoch 4/10 | Train Loss: 0.4981 | Train Accuracy: 0.7439 | Val Loss: 0.4651 | Val Accuracy: 0.7576 | Val Precision: 0.8571 | Val Recall: 0.1010 | Val F1: 0.1807 | Val AUC: 0.8378 | 
Epoch 5/10 | Train Loss: 0.4714 | Train Accuracy: 0.7720 | Val Loss: 0.4437 | Val Accuracy: 0.7754 | Val Precision: 0.6744 | Val Recall: 0.2929 | Val F1: 0.4085 | Val AUC: 0.8393 | 
Epoch 6/10 | Train Loss: 0.4536 | Train Accuracy: 0.7858 | Val Loss: 0.4326 | Val Accuracy

train_accuracy,▁▄▄▅▆▇████
train_loss,█▆▄▃▂▂▁▁▁▁
val_accuracy,▁▁▁▃▅▇████
val_auc,▁▇▇███████
val_f1,▁▁▁▃▆█████
val_loss,█▆▄▃▂▁▁▁▁▁
val_precision,▁▁▁█▇▇▆▆▆▆
val_recall,▁▁▁▂▅▇████
model_path,models/model_99fldt8...
train_accuracy,0.79313
train_loss,0.43665


## Fine Tunning

In [138]:
#Cargo el mejor modelo según el sweep
checkpoint = torch.load(
    "/home/jaume/ConquerX/Mis_apuntes/Pytorch/ejercicio_redes/Teleco_Customer_Churn/models/model_ul4slcuu.pt",
    map_location="cpu"
)

#Configuracion del modelo
model_config = checkpoint["valores_config"]

/tmp/ipykernel_10636/2295437087.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(


In [139]:
# Reconstruyo el modelo con los mejores parametros del modelo ya entrenado
new_model = ChurnModel(
    input_size=model_config["input_size"],
    hidden_size_1=model_config["hidden_size_1"],
    hidden_size_2=model_config["hidden_size_2"],
    dropout=model_config["dropout"]
)

criterion = BCEWithLogitsLoss()

# Cargo los pesos del modelo preentrenado
new_model.load_state_dict(checkpoint["model_state_dict"])


#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
new_model = new_model.to("cpu")
new_model.eval()

for name, param in new_model.named_parameters():
    print(name, param.shape, param.requires_grad)

network.0.weight torch.Size([32, 45]) True
network.0.bias torch.Size([32]) True
network.3.weight torch.Size([64, 32]) True
network.3.bias torch.Size([64]) True
network.6.weight torch.Size([1, 64]) True
network.6.bias torch.Size([1]) True


In [140]:
#Congelo las capas que no quiero entrenar
for param in new_model.network[0].parameters():
    param.requires_grad = False

for param in new_model.network[3].parameters():
    param.requires_grad = False

In [141]:
fine_tune_optimizer = Adam(
    filter(lambda param: param.requires_grad,new_model.parameters()),
    lr=model_config["lr"] / 10,
    weight_decay=model_config["weight_decay"]    
)

In [142]:
#Comprobacion de la congelacion de capas
for name, param in new_model.named_parameters():
    print(name, param.requires_grad)

network.0.weight False
network.0.bias False
network.3.weight False
network.3.bias False
network.6.weight True
network.6.bias True


In [ ]:
#Entreno la ultima capa

finetune_history = train_model(
    model=new_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=fine_tune_optimizer,
    device="cpu",
    epochs=3
    
)

Epoch 1/3 | Train Loss: 0.4335 | Train Accuracy: 0.7860 | Val Loss: 0.4209 | Val Accuracy: 0.8039 | Val Precision: 0.6808 | Val Recall: 0.4882 | Val F1: 0.5686 | Val AUC: 0.8422 | 
Epoch 2/3 | Train Loss: 0.4239 | Train Accuracy: 0.7987 | Val Loss: 0.4215 | Val Accuracy: 0.8048 | Val Precision: 0.6875 | Val Recall: 0.4815 | Val F1: 0.5663 | Val AUC: 0.8426 | 
Epoch 3/3 | Train Loss: 0.4244 | Train Accuracy: 0.7887 | Val Loss: 0.4218 | Val Accuracy: 0.8057 | Val Precision: 0.6908 | Val Recall: 0.4815 | Val F1: 0.5675 | Val AUC: 0.8428 | 


In [ ]:
# Cargo el modelo original
model_path = "/home/jaume/ConquerX/Mis_apuntes/Pytorch/ejercicio_redes/Teleco_Customer_Churn/models/model_ul4slcuu.pt"
checkpoint = torch.load(model_path, map_location="cpu")
model_config = checkpoint["valores_config"]

base_model = ChurnModel(
    input_size=model_config["input_size"],
    hidden_size_1=model_config["hidden_size_1"],
    hidden_size_2=model_config["hidden_size_2"],
    dropout=model_config["dropout"]
)

base_model.load_state_dict(checkpoint["model_state_dict"])
base_model = base_model.to("cpu")
base_model.eval()

/tmp/ipykernel_10636/3376924423.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location="cpu")


ChurnModel(
  (network): Sequential(
    (0): Linear(in_features=45, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.4, inplace=False)
    (3): Linear(in_features=32, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.4, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [ ]:
#Extraigo las metricas de los modelos

base_metrics = evaluate(
    model=base_model,
    data_loader=val_loader,
    criterion=criterion,
    device="cpu"
)

finetuned_metrics = evaluate(
    model=new_model,
    data_loader=val_loader,
    criterion=criterion,
    device="cpu"
)

In [ ]:
#Comparo las metricas
metric_names = ["loss", "accuracy", "precision", "recall", "f1", "auc"]

for name, base_value, fine_value in zip(metric_names, base_metrics, finetuned_metrics):
    print(f"{name}: base={base_value:.4f} | fine-tuned={fine_value:.4f}")

loss: base=0.4194 | fine-tuned=0.4218
accuracy: base=0.8066 | fine-tuned=0.8057
precision: base=0.6724 | fine-tuned=0.6908
recall: base=0.5253 | fine-tuned=0.4815
f1: base=0.5898 | fine-tuned=0.5675
auc: base=0.8430 | fine-tuned=0.8428


### El modelo no ha mejorado

## Fase de Test

In [155]:
base_model.eval()
test_metrics = evaluate(
    model=base_model,
    data_loader=test_loader,
    criterion=criterion,
    device="cpu"
)

In [156]:
metric_names = ["loss", "accuracy", "precision", "recall", "f1", "auc"]

for name, value in zip(metric_names, test_metrics):
    print(f"{name}: {value:.4f}")

loss: 0.4345
accuracy: 0.7924
precision: 0.6325
recall: 0.5148
f1: 0.5676
auc: 0.8322
